# STEP 1: CVE Data Ingestion & Reset Pipeline

**Purpose**: Run a reproducible ingestion workflow for resetting, loading, enriching, and validating the CVE database.

**What this notebook does**:
1. **Status Check** - View current database and cache status
2. **Reset Options** - Clear database, cache, or both
3. **Data Ingestion** - Fetch CVEs from NVD API
4. **Enrichment** - Add KEV, EPSS, Healthcare, ATT&CK, CHPL data
5. **Validation** - Verify data quality and completeness

**Prerequisites**:
- Set `NVD_API_KEY` in .env file (optional but recommended for faster fetching)
- Ensure all required cache directories exist

---

## 1. Setup & Imports

Initialize project paths, environment variables, and required modules for database access, enrichment, and analysis.

In [62]:
import sys
import os
import sqlite3
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Add project root to path
project_root = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(project_root))

# Pipeline functions (ingestion, enrichment, validation, reset)
# Full implementations are in src/utils/pipeline_utils.py
from src.utils.pipeline_utils import (
    check_current_status,
    reset_database, reset_cache, reset_all,
    fetch_cves_by_date,
    enrich_all_cves,
    validate_data_quality,
    export_enriched_data,
)
from config.settings import settings

print(f'[OK] Project root: {project_root}')
print(f'[OK] Database path: {settings.get_database_path()}')
print('[OK] Imports successful')


[OK] Project root: /Users/vinayksharma/AirDnd/cti_recommender
[OK] Database path: /Users/vinayksharma/AirDnd/cti_recommender/data/cve_database.db
[OK] Imports successful


## 2. Current Status Check

View current state of database and cache files

In [63]:
# See: src/utils/pipeline_utils.check_current_status
check_current_status(project_root)


CURRENT DATA STATUS

[STATS] Database: /Users/vinayksharma/AirDnd/cti_recommender/data/cve_database.db
   Size: 342.53 MB
2026-03-28 20:51:01 - src.core.cve_database - INFO - Connected to database
2026-03-28 20:51:01 - src.core.cve_database - INFO - Database schema created/verified
   Total CVEs: 226,320
   Enriched: 226,320
   Date range: 2018-01-01T00:29:00.213 to 2025-12-31T23:15:42.413

   Enrichment Signals:
      KEV (exploited): 1,179
      CHPL certified:  5,107
      Healthcare:      2,009
      ATT&CK mapped:   83,574

 Cache Directories:
   cache/nvd: 0 files (0.00 MB)
   cache/epss: 1 files (16.45 MB)
   cache/kev: 0 files (0.00 MB)
   cache/attack: 0 files (0.00 MB)
   cache/chpl: 2 files (4.03 MB)

 Enhanced Features (for Model Training):
   features_enhanced_latest.csv: 56 features (69.8 MB)
   [OK] Enhanced features ready for STEP_2 & STEP_3



## 3. Reset Options

Choose controlled reset actions for database and caches before a fresh ingestion run.

[WARN] **Warning**: These operations are destructive and cannot be undone!

In [64]:
# Reset functions are in src/utils/pipeline_utils.py
# Uncomment to run:
# reset_database(project_root, confirm=True)
# reset_cache(project_root, cache_type='epss', confirm=True)
# reset_all(project_root, confirm=True)

print('Reset functions loaded. Use with confirm=True to execute.')


Reset functions loaded. Use with confirm=True to execute.


## 4. Fetch CVEs from NVD

Fetch CVE data for a specific date range

In [65]:
# See: src/utils/pipeline_utils.fetch_cves_by_date

end_date = datetime.now(timezone.utc)
start_date = end_date - timedelta(days=30)

print(f"Ready to fetch CVEs from {start_date.strftime('%Y-%m-%d')} to {end_date.strftime('%Y-%m-%d')}")
print("\nTo fetch, run:")
print(f"df = fetch_cves_by_date(project_root, '{start_date.strftime('%Y-%m-%d')}', '{end_date.strftime('%Y-%m-%d')}')")


Ready to fetch CVEs from 2026-02-26 to 2026-03-28

To fetch, run:
df = fetch_cves_by_date(project_root, '2026-02-26', '2026-03-28')


### Quick Fetch Options

Uncomment one of the following to fetch data:

In [66]:
# Option 1: Fetch last 7 days
# end = datetime.now(timezone.utc).strftime('%Y-%m-%d')
# start = (datetime.now(timezone.utc) - timedelta(days=7)).strftime('%Y-%m-%d')
# df = fetch_cves_by_date(start, end)

# Option 2: Fetch last 30 days
# end = datetime.now(timezone.utc).strftime('%Y-%m-%d')
# start = (datetime.now(timezone.utc) - timedelta(days=30)).strftime('%Y-%m-%d')
# df = fetch_cves_by_date(start, end)

# Option 3: Fetch specific date range
# df = fetch_cves_by_date('2024-01-01', '2024-01-31')

# Option 4: Fetch 2025 data (current year)
# df = fetch_cves_by_date('2025-01-01', '2025-12-31')

print("Uncomment one of the options above to fetch CVEs")

Uncomment one of the options above to fetch CVEs


## 5. Enrich CVEs with Multi-Source Data

Add KEV, EPSS, Healthcare, ATT&CK, and CHPL enrichments

In [67]:
# See: src/utils/pipeline_utils.enrich_all_cves
# Runs steps: KEV, EPSS, Healthcare, Curated, ATT&CK, CHPL
print('Enrichment pipeline loaded. Run: enrich_all_cves(project_root)')


Enrichment pipeline loaded. Run: enrich_all_cves(project_root)


In [68]:
# Uncomment to run full enrichment
# enrich_all_cves()

## 6. Validation & Quality Checks

In [69]:
# See: src/utils/pipeline_utils.validate_data_quality
validate_data_quality(project_root)


2026-03-28 20:51:02 - src.core.cve_database - INFO - Connected to database
2026-03-28 20:51:02 - src.core.cve_database - INFO - Database schema created/verified
DATA QUALITY VALIDATION

[STATS] Coverage:
   Total CVEs: 226,320
   Enriched:   226,320 (100.0%)

[TARGET] Enrichment Signals:
   KEV (exploited)                 1,179 ( 0.52%)
   Healthcare-related              2,009 ( 0.89%)
   ATT&CK mapped                  83,574 (36.93%)
   CHPL certified                  5,107 ( 2.26%)
   Curated breaches                   52 ( 0.02%)
   EPSS scores                    226,320 (100.00%)

 High-Value CVEs (multiple signals):
   KEV + Healthcare: 5
   CHPL + Healthcare: 120
   ATT&CK + Healthcare: 874
   KEV + ATT&CK + Healthcare: 1

 CVE Distribution by Year (last 10):
   2025: 49,972
   2024: 40,704
   2023: 30,949
   2022: 26,431
   2021: 21,950
   2020: 19,222
   2019: 18,938
   2018: 18,154

[WARN]  CVSS Severity Distribution:
   Critical (9.0-10.0)       25,231 (11.15%)
   High (7.0-8

## 7. Quick Analysis Queries

In [70]:
# Healthcare CVE Analysis Queries
db_path = project_root / "data" / "cve_database.db"
conn = sqlite3.connect(db_path)

# Query 1: KEV + Healthcare (Critical!)
print("=" * 100)
print("CRITICAL: KEV-Listed Healthcare CVEs (Actively Exploited)")
print("=" * 100)
query1 = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    e.epss_score,
    e.healthcare_score,
    SUBSTR(c.description, 1, 80) as description_short
FROM cves c
JOIN enrichments e ON c.cve_id = e.cve_id
WHERE e.kev_flag = 1 AND e.is_healthcare = 1
ORDER BY c.cvss DESC, e.epss_score DESC
"""
kev_healthcare = pd.read_sql_query(query1, conn)
print(f"Found {len(kev_healthcare)} KEV + Healthcare CVEs\n")
display(kev_healthcare)

# Query 2: Top 20 Healthcare CVEs by CVSS
print("\n" + "=" * 100)
print("TOP 20: Highest Severity Healthcare CVEs")
print("=" * 100)
query2 = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    e.epss_score,
    e.healthcare_score,
    e.chpl_flag,
    e.kev_flag,
    SUBSTR(c.description, 1, 80) as description_short
FROM cves c
JOIN enrichments e ON c.cve_id = e.cve_id
WHERE e.is_healthcare = 1
ORDER BY c.cvss DESC, e.healthcare_score DESC
LIMIT 20
"""
top_healthcare = pd.read_sql_query(query2, conn)
display(top_healthcare)

# Query 3: CHPL Certified Product CVEs
print("\n" + "=" * 100)
print("TOP 20: CHPL Certified Health IT Product Vulnerabilities")
print("=" * 100)
query3 = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    e.epss_score,
    e.healthcare_score,
    e.kev_flag,
    SUBSTR(c.description, 1, 80) as description_short
FROM cves c
JOIN enrichments e ON c.cve_id = e.cve_id
WHERE e.chpl_flag = 1
ORDER BY c.cvss DESC, e.epss_score DESC
LIMIT 20
"""
chpl_cves = pd.read_sql_query(query3, conn)
print(f"Showing top 20 of {len(chpl_cves)} CHPL-related CVEs\n")
display(chpl_cves)

# Query 4: Multi-Signal Healthcare CVEs
print("\n" + "=" * 100)
print("HIGH VALUE: Healthcare CVEs with Multiple Risk Signals")
print("=" * 100)
query4 = """
SELECT 
    c.cve_id,
    c.published,
    c.cvss,
    e.epss_score,
    e.healthcare_score,
    e.kev_flag,
    e.chpl_flag,
    e.attack_flag,
    CASE 
        WHEN e.kev_flag + e.chpl_flag + e.attack_flag >= 2 THEN 'High'
        ELSE 'Medium'
    END as priority,
    SUBSTR(c.description, 1, 80) as description_short
FROM cves c
JOIN enrichments e ON c.cve_id = e.cve_id
WHERE e.is_healthcare = 1 
  AND (e.kev_flag = 1 OR e.chpl_flag = 1 OR e.attack_flag = 1)
ORDER BY (e.kev_flag + e.chpl_flag + e.attack_flag) DESC, c.cvss DESC
LIMIT 25
"""
multi_signal = pd.read_sql_query(query4, conn)
display(multi_signal)

conn.close()

CRITICAL: KEV-Listed Healthcare CVEs (Actively Exploited)
Found 5 KEV + Healthcare CVEs



,cve_id,published,cvss,epss_score,healthcare_score,description_short
0,CVE-2023-43208,2023-10-26T17:15:09.033,9.8,0.94416,0.7,NextGen Healthcare Mirth Connect before versio...
1,CVE-2020-15505,2020-07-07T02:15:10.613,9.8,0.94388,0.2,A remote code execution vulnerability in Mobil...
2,CVE-2018-19410,2018-11-21T16:29:00.347,9.8,0.91753,0.2,PRTG Network Monitor before 18.2.40.1683 allow...
3,CVE-2020-10181,2020-03-11T16:15:12.007,9.8,0.20551,0.5,goform/formEMR30 in Sumavision Enhanced Multim...
4,CVE-2018-9276,2018-07-02T16:29:00.600,7.2,0.81539,0.2,An issue was discovered in PRTG Network Monito...



TOP 20: Highest Severity Healthcare CVEs


,cve_id,published,cvss,epss_score,healthcare_score,chpl_flag,kev_flag,description_short
0,CVE-2025-52572,2025-06-24T21:15:25.463,10.0,0.00407,0.5,0,0,"Hikka, a Telegram userbot, has vulnerability a..."
1,CVE-2025-29009,2025-07-16T12:15:24.680,10.0,0.00070,0.5,0,0,Unrestricted Upload of File with Dangerous Typ...
2,CVE-2025-22609,2025-01-24T17:15:15.100,10.0,0.00559,0.5,0,0,Coolify is an open-source and self-hostable to...
3,CVE-2024-48967,2024-11-14T22:15:17.927,10.0,0.00206,0.5,0,0,The ventilator and the Service PC lack suffici...
4,CVE-2024-48966,2024-11-14T22:15:17.727,10.0,0.00184,0.5,0,0,The software tools used by service personnel t...
5,CVE-2019-5644,2019-11-06T19:15:12.547,10.0,0.01914,0.5,0,0,Computing For Good's Basic Laboratory Informat...
6,CVE-2019-5617,2019-11-06T19:15:12.233,10.0,0.01914,0.5,0,0,Computing For Good's Basic Laboratory Informat...
7,CVE-2019-10959,2019-06-13T21:29:15.817,10.0,0.01062,0.5,0,0,"BD Alaris Gateway Workstation Versions, 1.1.3 ..."
8,CVE-2025-42890,2025-11-11T01:15:37.820,10.0,0.00097,0.2,0,0,SQL Anywhere Monitor (Non-GUI) baked credentia...
9,CVE-2025-39380,2025-05-19T20:15:24.400,10.0,0.00066,0.2,0,0,Unrestricted Upload of File with Dangerous Typ...



TOP 20: CHPL Certified Health IT Product Vulnerabilities
Showing top 20 of 20 CHPL-related CVEs



,cve_id,published,cvss,epss_score,healthcare_score,kev_flag,description_short
0,CVE-2019-11510,2019-05-08T17:29:00.630,10.0,0.94380,0.0,1,In Pulse Secure Pulse Connect Secure (PCS) 8.2...
1,CVE-2021-22893,2021-04-23T17:15:08.127,10.0,0.93378,0.0,1,Pulse Connect Secure 9.0R3/9.1R1 and higher is...
2,CVE-2022-25226,2022-04-18T17:15:16.693,10.0,0.77082,0.0,0,ThinVNC version 1.0b1 allows an unauthenticate...
3,CVE-2019-11061,2019-08-29T01:15:10.930,10.0,0.11613,0.0,0,A broken access control vulnerability in HG100...
4,CVE-2022-43605,2023-03-16T21:15:11.203,10.0,0.07387,0.0,0,An out-of-bounds write vulnerability exists in...
5,CVE-2024-42472,2024-08-15T19:15:19.233,10.0,0.06541,0.0,0,Flatpak is a Linux application sandboxing and ...
6,CVE-2022-43604,2023-03-16T21:15:11.123,10.0,0.03966,0.0,0,An out-of-bounds write vulnerability exists in...
7,CVE-2022-37968,2022-10-11T19:15:12.030,10.0,0.03676,0.0,0,Microsoft has identified a vulnerability affec...
8,CVE-2020-6932,2020-08-12T13:15:10.833,10.0,0.03626,0.0,0,An information disclosure and remote code exec...
9,CVE-2023-37470,2023-08-04T16:15:09.610,10.0,0.03351,0.0,0,Metabase is an open-source business intelligen...



HIGH VALUE: Healthcare CVEs with Multiple Risk Signals


,cve_id,published,cvss,epss_score,healthcare_score,kev_flag,chpl_flag,attack_flag,priority,description_short
0,CVE-2024-36543,2024-06-17T19:15:58.353,9.8,0.00124,0.5,0,1,1,High,Incorrect access control in the Kafka Connect ...
1,CVE-2023-43208,2023-10-26T17:15:09.033,9.8,0.94416,0.7,1,1,0,High,NextGen Healthcare Mirth Connect before versio...
2,CVE-2021-27410,2021-06-11T17:15:10.770,9.8,0.00552,0.2,0,1,1,High,The affected product is vulnerable to an out-o...
3,CVE-2024-48970,2024-11-14T22:15:18.137,9.3,0.00065,0.5,0,1,1,High,The ventilator's microcontroller lacks memory ...
4,CVE-2021-22156,2021-08-17T19:15:08.057,9.0,0.01277,0.2,0,1,1,High,An integer overflow vulnerability in the callo...
5,CVE-2019-11875,2019-05-24T16:29:00.437,8.8,0.00339,0.5,0,1,1,High,In AutomateAppCore.dll in Blue Prism Robotic P...
6,CVE-2017-9388,2019-06-17T17:15:10.537,8.8,0.01017,0.5,0,1,1,High,An issue was discovered on Vera VeraEdge 1.7.1...
7,CVE-2017-9384,2019-06-17T18:15:10.627,8.8,0.01017,0.5,0,1,1,High,An issue was discovered on Vera VeraEdge 1.7.1...
8,CVE-2017-12712,2018-04-25T13:29:00.227,8.8,0.00515,0.8,0,1,1,High,The authentication algorithm in Abbott Laborat...
9,CVE-2024-32030,2024-06-19T17:15:57.863,8.1,0.81722,0.2,0,1,1,High,Kafka UI is an Open-Source Web UI for Apache K...


## 8. Export Data for Analysis

In [71]:
# See: src/utils/pipeline_utils.export_enriched_data
# Uncomment to export:
# export_enriched_data(project_root)


## 9. Summary

**Complete Workflow**:

1. **Check status**: `check_current_status()`
2. **Reset (if needed)**: `reset_all(confirm=True)`
3. **Fetch CVEs**: `fetch_cves_by_date('2024-01-01', '2024-12-31')`
4. **Enrich data**: `enrich_all_cves()`
5. **Validate**: `validate_data_quality()`
6. **Export**: `export_enriched_data()`

**Next Steps**:
- Feature engineering notebook
- Model training notebook
- Recommendation generation

---

## 10. Schema & Data Integrity Validation

This section adds explicit schema checks, key uniqueness checks, and null/format diagnostics to improve auditability and reproducibility.

In [72]:
import re
from datetime import datetime

print('\n' + '='*70)
print('SCHEMA & DATA INTEGRITY AUDIT')
print('='*70)

def _existing_table(conn_obj, candidates):
    for t in candidates:
        row = conn_obj.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (t,)).fetchone()
        if row:
            return t
    return None

try:
    if 'conn' not in globals() or conn is None:
        raise ValueError('Database connection `conn` is not available in this notebook kernel.')

    table_candidates = ['cves', 'cve_enhanced', 'vulnerabilities', 'processed_cves']
    table_name = _existing_table(conn, table_candidates)
    if table_name is None:
        raise ValueError(f'None of expected tables found: {table_candidates}')

    schema_rows = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
    schema_cols = [r[1] for r in schema_rows]
    print(f"[OK] Target table: {table_name}")
    print(f"[INFO] Column count: {len(schema_cols)}")

    required_cols = ['cve_id', 'published']
    missing_required = [c for c in required_cols if c not in schema_cols]
    if missing_required:
        print(f"[WARN] Missing required columns: {missing_required}")
    else:
        print('[OK] Required schema columns are present')

    if 'cve_id' in schema_cols:
        q_dup = f"""
            SELECT COUNT(*)
            FROM (
                SELECT cve_id
                FROM {table_name}
                GROUP BY cve_id
                HAVING COUNT(*) > 1
            ) x
        """
        dup_count = int(conn.execute(q_dup).fetchone()[0])
        print(f"[INFO] Duplicate CVE IDs: {dup_count}")

        q_null = f"SELECT COUNT(*) FROM {table_name} WHERE cve_id IS NULL OR TRIM(cve_id) = ''"
        null_count = int(conn.execute(q_null).fetchone()[0])
        print(f"[INFO] Empty/NULL CVE IDs: {null_count}")

        sample_q = f"SELECT cve_id FROM {table_name} WHERE cve_id IS NOT NULL LIMIT 5000"
        ids = [r[0] for r in conn.execute(sample_q).fetchall()]
        patt = re.compile(r'^CVE-\d{4}-\d{4,}$')
        bad_fmt = sum(1 for cid in ids if not patt.match(str(cid).strip()))
        print(f"[INFO] CVE format violations in sample (n={len(ids)}): {bad_fmt}")

    audit_summary = {
        'audit_time_utc': datetime.utcnow().isoformat(timespec='seconds'),
        'table_name': table_name,
        'column_count': len(schema_cols),
        'missing_required_columns': missing_required,
        'duplicate_cve_ids': dup_count if 'cve_id' in schema_cols else None,
        'null_or_empty_cve_ids': null_count if 'cve_id' in schema_cols else None,
        'sample_bad_cve_format': bad_fmt if 'cve_id' in schema_cols else None,
    }
    print('\n[OK] Ingestion integrity audit complete')
    print(audit_summary)

except Exception as e:
    print(f"[ERROR] Schema/integrity audit failed: {e}")


SCHEMA & DATA INTEGRITY AUDIT
[ERROR] Schema/integrity audit failed: Cannot operate on a closed database.
